# reshape-back — worked example 2: reshape_back when merging two axes

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `reshape-back`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Merging axes — e.g. `(B, C, H, W) -> (B, C, H*W)` — is still a view-style reshape, so the backward rule is unchanged: reshape `grad_out` back to the original `x.shape`. The same primitive `reshape_back` handles split and merge alike because both are pure index remappings.

## Worked solution

The forward merges the last two spatial axes of a `(2, 3, 4, 5)` tensor into `(2, 3, 20)`. `reshape_back` reshapes the `(2, 3, 20)` gradient back to `(2, 3, 4, 5)`. The merge and its inverse split are duals, and `x.shape` already encodes the target, so no special handling is needed. We confirm the result equals autograd's gradient on the equivalent forward, demonstrating that one back fn covers the whole reshape family.

In [ ]:
Tensor = t.Tensor


def reshape_back(grad_out: Tensor, out: Tensor, x: Tensor, new_shape: tuple) -> Tensor:
    return grad_out.reshape(x.shape)


t.manual_seed(1)
x = t.randn(2, 3, 4, 5)
out = x.reshape(2, 3, 20)
grad_out = t.randn(2, 3, 20)
grad_x = reshape_back(grad_out, out, x, (2, 3, 20))
print('grad_x shape:', tuple(grad_x.shape))

xg = x.clone().requires_grad_(True)
xg.reshape(2, 3, 20).backward(grad_out)
print('matches autograd:', t.allclose(grad_x, xg.grad))